# Public tech-support ticket preview and synthetic support mining with DSPy RLM

By default, this notebook reads the bundled Apache-2.0 public `tech_support_dataset.csv` and classifies a bounded sample of issue/response/status **records** without writing anything to Snowflake. The dataset has no timestamps or customer confirmation; a publisher `Resolved` label is not proof that a proposed response fixed the issue. Set `SOURCE = 'seeded_threads'` after running `setup.sql` to exercise the separate publishing workflow with the original three explicitly staged threads. Put `support.py`, `snowflake_lm.py`, and the CSV alongside this notebook. It runs on Snowflake Container Runtime: the active Snowpark session calls `AI_COMPLETE` with no PAT or model HTTP endpoint; RLM-generated code runs only in `dspy.PythonInterpreter` (Deno/Pyodide), without a Snowflake session. Interpreter startup and model subcalls incur costs.

In [ ]:
# dspy 3.5 (needed for dspy.lm15) requires a newer cloudpickle than Snowpark pins; fine here since no UDFs are registered.
# The [deno] extra provides the Deno 2.x runtime that dspy.PythonInterpreter needs.
%pip install -q "dspy[deno]>=3.5"

In [ ]:
import json
import dspy
from pydantic import ValidationError
from snowflake import snowpark
from snowflake_lm import SnowflakeCompleteEngine
from support import Classification, ordered_threads, public_ticket_sample, validated_issue, publish_chunks, serialize_thread, validate_evidence

# The notebook's active Snowpark session is the only Snowflake connection.
session = snowpark.context.get_active_session()
if not hasattr(dspy, 'RLM') or not hasattr(dspy, 'PythonInterpreter'):
    raise RuntimeError('Install a DSPy release supporting RLM and PythonInterpreter')
with dspy.PythonInterpreter() as sandbox:
    sandbox('print(1 + 1)')
model = dspy.LM('custom/snowflake-ai-complete', engine=SnowflakeCompleteEngine(session), max_tokens=1500, cache=False, num_retries=0)
dspy.configure(lm=model)
print('RLM sandbox and active-session AI_COMPLETE configured')

In [ ]:
SOURCE = 'public_tickets'  # Change to 'seeded_threads' only for the write/publish demo.
if SOURCE not in ('public_tickets', 'seeded_threads'):
    raise ValueError('Unknown source')
if SOURCE == 'public_tickets':
    public_records = public_ticket_sample('tech_support_dataset.csv', per_status=2)
    print('Publisher statuses:', [record['Issue_Status'] for record in public_records])
else:
    rows = session.sql('SELECT THREAD_ID, MESSAGE_ID, POSTED_AT, BODY FROM RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES ORDER BY THREAD_ID, POSTED_AT, MESSAGE_ID LIMIT 100').collect()

In [ ]:
from typing import Literal

class ClassifyThread(dspy.Signature):
    """Read ordered support messages; resolved requires an explicit successful fix. Cite source message_id strings."""
    messages: str = dspy.InputField()
    status: str = dspy.OutputField(desc='Exactly one of resolved, unresolved, escalated')
    evidence_ids: list[str] = dspy.OutputField(desc='Source message IDs supporting the outcome')

class ExtractIssue(dspy.Signature):
    """Extract a verified technical fix from a resolved thread. Never invent steps or message IDs."""
    messages: str = dspy.InputField()
    title: str = dspy.OutputField()
    symptom: str = dspy.OutputField()
    resolution: str = dspy.OutputField()
    evidence_ids: list[str] = dspy.OutputField()

class ClassifyPublicTicket(dspy.Signature):
    """Classify a public single-turn ticket for review, not publication. A suggested step and publisher label are not customer confirmation."""
    ticket: str = dspy.InputField()
    # Literal makes DSPy enforce the same closed set that support.Classification validates.
    status: Literal['resolved', 'unresolved', 'escalated'] = dspy.OutputField(desc='Without explicit customer confirmation do not claim resolved; use unresolved when review is needed')
    rationale: str = dspy.OutputField(desc='Brief explanation tied to the issue, response, and publisher status; no invented follow-up')

threads = {}
if SOURCE == 'public_tickets':
    public_classifier = dspy.RLM(ClassifyPublicTicket, max_iters=4, max_llm_calls=5, max_output_chars=1200, interpreter_factory=dspy.PythonInterpreter)
else:
    classifier = dspy.RLM(ClassifyThread, max_iters=5, max_llm_calls=6, max_output_chars=2000, interpreter_factory=dspy.PythonInterpreter)
    extractor = dspy.RLM(ExtractIssue, max_iters=6, max_llm_calls=8, max_output_chars=2000, interpreter_factory=dspy.PythonInterpreter)
    threads = ordered_threads(rows)
    if len(threads) > 10:
        raise ValueError('Limit demo to 10 threads')
    print('Thread count:', len(threads))

In [ ]:
# Public source: preview classification only. Never persist a proposed fix from this CSV.
if SOURCE == 'public_tickets':
    for record in public_records:
        ticket = {key: record[key] for key in ('Conversation_ID', 'Customer_Issue', 'Tech_Response', 'Issue_Category', 'Issue_Status')}
        outcome = public_classifier(ticket=json.dumps(ticket, ensure_ascii=True))
        label = Classification.model_validate({'status': outcome.status, 'evidence_ids': [record['Conversation_ID']]})
        if label.status == 'resolved':
            raise ValueError(f"Unconfirmed public ticket cannot be marked resolved: {record['Conversation_ID']}")
        rationale = str(outcome.rationale).strip()
        if not rationale or len(rationale) > 600:
            raise ValueError('Invalid public classification rationale')
        print(record['Conversation_ID'], 'publisher:', record['Issue_Status'], 'preview:', label.status, rationale)

In [ ]:
SOURCE = 'seeded_threads'

In [ ]:
%%sql -r dataframe_2
 SELECT 'msgs' t, COUNT(*) n FROM RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES
    UNION ALL SELECT 'results', COUNT(*) FROM RACO_RLM_DEMO.PUBLIC.THREAD_RESULTS
    UNION ALL SELECT 'issues', COUNT(*) FROM RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES
    UNION ALL SELECT 'chunks', COUNT(*) FROM RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS;


In [ ]:
%%sql -r dataframe_1
-- Run only with an authorized write role after reviewing the namespace.
CREATE DATABASE IF NOT EXISTS RACO_RLM_DEMO;
CREATE SCHEMA IF NOT EXISTS RACO_RLM_DEMO.PUBLIC;
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES (
  THREAD_ID VARCHAR, MESSAGE_ID VARCHAR, POSTED_AT TIMESTAMP_NTZ, BODY VARCHAR
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.THREAD_RESULTS (
  THREAD_ID VARCHAR, STATUS VARCHAR, EVIDENCE_IDS VARIANT
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES (
  THREAD_ID VARCHAR, TITLE VARCHAR, DOCUMENT VARCHAR, EVIDENCE_IDS VARIANT
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS (
  CHUNK_ID VARCHAR, THREAD_ID VARCHAR, TITLE VARCHAR, CHUNK_TEXT VARCHAR, EVIDENCE_IDS VARIANT
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.VISION_KEYPOINTS (
  RUN_ID VARCHAR, POINT_INDEX INTEGER, PAYLOAD VARIANT
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.WEIGHT_EDIT_RUNS (
  RUN_ID VARCHAR, RESULT VARIANT
);
CREATE TABLE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.EVAL_QUESTIONS (
  INPUT_QUERY VARCHAR, GROUND_TRUTH VARIANT
);
CREATE STAGE IF NOT EXISTS RACO_RLM_DEMO.PUBLIC.MODEL_ASSETS ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE');

INSERT INTO RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES (THREAD_ID, MESSAGE_ID, POSTED_AT, BODY)
SELECT v.column1, v.column2, TO_TIMESTAMP_NTZ(v.column3), v.column4 FROM VALUES
 ('T-100', 'M-100', '2026-01-10 09:00:00', 'Scheduled task fails with missing warehouse after migration.'),
 ('T-100', 'M-101', '2026-01-10 09:02:00', 'Use a warehouse with USAGE granted to the task owner role and set it on the task.'),
 ('T-100', 'M-102', '2026-01-10 09:07:00', 'Confirmed: task succeeded after setting the authorized warehouse.'),
 ('T-200', 'M-200', '2026-01-11 11:00:00', 'Search service is missing expected results after load.'),
 ('T-200', 'M-201', '2026-01-11 11:05:00', 'Checking refresh history; no fix confirmed yet.'),
 ('T-300', 'M-300', '2026-01-12 12:00:00', 'Connector repeatedly fails authentication.'),
 ('T-300', 'M-301', '2026-01-12 12:03:00', 'Escalated to identity team for investigation; no resolution yet.') v
WHERE NOT EXISTS (
  SELECT 1 FROM RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES m WHERE m.MESSAGE_ID = v.column2
);

INSERT INTO RACO_RLM_DEMO.PUBLIC.EVAL_QUESTIONS (INPUT_QUERY, GROUND_TRUTH)
SELECT v.column1, PARSE_JSON(v.column2) FROM VALUES
 ('What fixed the scheduled task missing-warehouse error?', '{"ground_truth_output":"The task succeeded after an authorized warehouse was set; the task owner needs USAGE on that warehouse. Cite the resolved thread T-100, not an unresolved thread.","ground_truth_invocations":[{"tool_name":"support_search","tool_input":"Search for scheduled task warehouse failure resolution"}]}'),
 ('Was the Search service refresh issue resolved?', '{"ground_truth_output":"No validated resolution for that issue appears in this knowledge base. Do not invent a fix.","ground_truth_invocations":[{"tool_name":"support_search"}]}'),
 ('How do I solve the connector authentication issue?', '{"ground_truth_output":"The thread was escalated with no confirmed resolution; do not give an invented fix.","ground_truth_invocations":[{"tool_name":"support_search"}]}') v
WHERE NOT EXISTS (
  SELECT 1 FROM RACO_RLM_DEMO.PUBLIC.EVAL_QUESTIONS q WHERE q.INPUT_QUERY = v.column1
);

In [ ]:
for thread_id, messages in threads.items():
    serialized = serialize_thread(messages)
    for attempt in range(2):
        try:
            outcome = classifier(messages=serialized)
            label = Classification.model_validate({'status': outcome.status, 'evidence_ids': outcome.evidence_ids})
            validate_evidence(label.evidence_ids, messages)
            issue_data = None
            if label.status == 'resolved':
                result = extractor(messages=serialized)
                issue_data = {field: getattr(result, field) for field in ('title', 'symptom', 'resolution', 'evidence_ids')}
            issue = validated_issue(label, issue_data, messages)
            chunks = publish_chunks(thread_id, issue) if issue is not None else []
            break
        except (ValueError, ValidationError, AttributeError) as error:
            if attempt == 1:
                raise RuntimeError(f'Validation failed for {thread_id}; nothing published') from error
            print('Retrying validation:', thread_id)
    session.sql('MERGE INTO RACO_RLM_DEMO.PUBLIC.THREAD_RESULTS target USING (SELECT ? AS THREAD_ID, ? AS STATUS, PARSE_JSON(?) AS EVIDENCE_IDS) source ON target.THREAD_ID = source.THREAD_ID WHEN MATCHED THEN UPDATE SET STATUS = source.STATUS, EVIDENCE_IDS = source.EVIDENCE_IDS WHEN NOT MATCHED THEN INSERT (THREAD_ID, STATUS, EVIDENCE_IDS) VALUES (source.THREAD_ID, source.STATUS, source.EVIDENCE_IDS)', params=[thread_id, label.status, json.dumps(label.evidence_ids)]).collect()
    if issue is None:
        session.sql('DELETE FROM RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS WHERE THREAD_ID = ?', params=[thread_id]).collect()
        session.sql('DELETE FROM RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES WHERE THREAD_ID = ?', params=[thread_id]).collect()
        print(thread_id, label.status, '(not published)')
        continue
    # Remove prior chunks for this thread so reruns never leave stale text.
    session.sql('DELETE FROM RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS WHERE THREAD_ID = ?', params=[thread_id]).collect()
    document = f'# {issue.title}\n\nSymptom: {issue.symptom}\n\nResolution: {issue.resolution}'
    session.sql('MERGE INTO RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES target USING (SELECT ? AS THREAD_ID, ? AS TITLE, ? AS DOCUMENT, PARSE_JSON(?) AS EVIDENCE_IDS) source ON target.THREAD_ID = source.THREAD_ID WHEN MATCHED THEN UPDATE SET TITLE = source.TITLE, DOCUMENT = source.DOCUMENT, EVIDENCE_IDS = source.EVIDENCE_IDS WHEN NOT MATCHED THEN INSERT (THREAD_ID, TITLE, DOCUMENT, EVIDENCE_IDS) VALUES (source.THREAD_ID, source.TITLE, source.DOCUMENT, source.EVIDENCE_IDS)', params=[thread_id, issue.title, document, json.dumps(issue.evidence_ids)]).collect()
    for chunk in chunks:
        session.sql('MERGE INTO RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS target USING (SELECT ? AS CHUNK_ID, ? AS THREAD_ID, ? AS TITLE, ? AS CHUNK_TEXT, PARSE_JSON(?) AS EVIDENCE_IDS) source ON target.CHUNK_ID = source.CHUNK_ID WHEN MATCHED THEN UPDATE SET CHUNK_TEXT = source.CHUNK_TEXT, EVIDENCE_IDS = source.EVIDENCE_IDS WHEN NOT MATCHED THEN INSERT (CHUNK_ID, THREAD_ID, TITLE, CHUNK_TEXT, EVIDENCE_IDS) VALUES (source.CHUNK_ID, source.THREAD_ID, source.TITLE, source.CHUNK_TEXT, source.EVIDENCE_IDS)', params=[chunk['chunk_id'], thread_id, issue.title, chunk['chunk_text'], json.dumps(chunk['evidence_ids'])]).collect()
    print(thread_id, 'validated and published')

## Cortex Sense scope (see the guide's "Scope a Cortex Sense Context" section)

This section **reviews** a narrow support-history scope. It does not create or build a Cortex Sense context.

- **Include, only if deployed and readable:** `SUPPORT_MESSAGES` (source messages), `THREAD_RESULTS` (classification status), `VALIDATED_ISSUES` (validated fixes) and `ISSUE_CHUNKS` (Search inputs) in `RACO_RLM_DEMO.PUBLIC`.
- **Exclude:** `VISION_KEYPOINTS`, `WEIGHT_EDIT_RUNS`, the public `tech_support_dataset.csv` (labels and suggestions, no customer confirmation) and unrelated account objects. Avoid an empty-rule, account-wide scope.
- **Grain:** one message per `MESSAGE_ID`, one seeded thread per `THREAD_ID`, one validated issue per resolved thread, and possibly many chunks per issue. Fix claims need source message IDs. Unresolved or escalated threads supply no confirmed fix.

The Sense workflow has **separate approval checkpoints**: (1) review the discovered scope, (2) approve the routing description, (3) only then persist and build. The cells below support checkpoint 1.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
NAMESPACE = 'RACO_RLM_DEMO.PUBLIC'
INCLUDE = ['SUPPORT_MESSAGES', 'THREAD_RESULTS', 'VALIDATED_ISSUES', 'ISSUE_CHUNKS']
EXCLUDE = ['VISION_KEYPOINTS', 'WEIGHT_EDIT_RUNS']

visible = {row['name'] for row in session.sql(f'SHOW TABLES IN SCHEMA {NAMESPACE}').collect()}
scope = {}
for table in INCLUDE:
    if table not in visible:
        scope[table] = 'missing: deploy before scoping'
        continue
    try:
        rows = session.sql(f'SELECT COUNT(*) AS N FROM {NAMESPACE}.{table}').collect()[0]['N']
        scope[table] = f'include ({rows} rows)'
    except Exception as error:
        scope[table] = f'not readable by {session.get_current_role()}: {error}'

services = session.sql(f"SHOW CORTEX SEARCH SERVICES LIKE 'SUPPORT_SEARCH' IN SCHEMA {NAMESPACE}").collect()
print('Role:', session.get_current_role())
print('Proposed include:')
for table, state in scope.items():
    print(f'  {NAMESPACE}.{table}: {state}')
print('Proposed exclude:', [f'{NAMESPACE}.{t}' for t in EXCLUDE if t in visible] + ['tech_support_dataset.csv (public, unconfirmed)'])
print('SUPPORT_SEARCH service:', 'deployed' if services else 'not deployed (search.sql is context, not proof)')

In [ ]:
%%sql -r sense_grain_checks
WITH m AS (SELECT * FROM RACO_RLM_DEMO.PUBLIC.SUPPORT_MESSAGES),
r AS (SELECT * FROM RACO_RLM_DEMO.PUBLIC.THREAD_RESULTS),
i AS (SELECT * FROM RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES),
c AS (SELECT * FROM RACO_RLM_DEMO.PUBLIC.ISSUE_CHUNKS),
cited AS (
  SELECT 'issue' src, i.THREAD_ID, e.value::VARCHAR MESSAGE_ID FROM i, LATERAL FLATTEN(i.EVIDENCE_IDS) e
  UNION ALL SELECT 'chunk', c.THREAD_ID, e.value::VARCHAR FROM c, LATERAL FLATTEN(c.EVIDENCE_IDS) e
  UNION ALL SELECT 'result', r.THREAD_ID, e.value::VARCHAR FROM r, LATERAL FLATTEN(r.EVIDENCE_IDS) e
)
SELECT 'One row per MESSAGE_ID' AS check_name, (SELECT COUNT(*) - COUNT(DISTINCT MESSAGE_ID) FROM m) AS violations
UNION ALL SELECT 'One result per THREAD_ID', (SELECT COUNT(*) - COUNT(DISTINCT THREAD_ID) FROM r)
UNION ALL SELECT 'Status in closed set', (SELECT COUNT(*) FROM r WHERE STATUS IS NULL OR STATUS NOT IN ('resolved', 'unresolved', 'escalated'))
UNION ALL SELECT 'One validated issue per THREAD_ID', (SELECT COUNT(*) - COUNT(DISTINCT THREAD_ID) FROM i)
UNION ALL SELECT 'Issues only for resolved threads', (SELECT COUNT(*) FROM i LEFT JOIN r ON r.THREAD_ID = i.THREAD_ID WHERE r.STATUS IS DISTINCT FROM 'resolved')
UNION ALL SELECT 'Chunks only for validated issues', (SELECT COUNT(*) FROM c LEFT JOIN i ON i.THREAD_ID = c.THREAD_ID WHERE i.THREAD_ID IS NULL)
UNION ALL SELECT 'Evidence IDs exist in same thread', (SELECT COUNT(*) FROM cited LEFT JOIN m ON m.THREAD_ID = cited.THREAD_ID AND m.MESSAGE_ID = cited.MESSAGE_ID WHERE m.MESSAGE_ID IS NULL)
UNION ALL SELECT 'Issues have non-empty evidence', (SELECT COUNT(*) FROM i WHERE COALESCE(ARRAY_SIZE(i.EVIDENCE_IDS), 0) = 0)

In [ ]:
# Proposed new cases from the guide's "Scope a Cortex Sense Context" section. Do NOT insert into EVAL_QUESTIONS
# (that table tests the Search-only agent) until reviewed against the deployed tools.
CANDIDATE_SENSE_EVALS = [
    {'question': 'What is a confirmed fix in this demo?',
     'expected_tool': 'support_context',
     'expected': 'Governed definition only if the built context defines it; otherwise abstain and inspect the ontology directly.'},
    {'question': 'Which messages confirm the scheduled-task warehouse fix?',
     'expected_tool': 'support_search',
     'expected': 'Cite T-100 with M-101 and M-102 as source records; a definition is not proof of success.'},
    {'question': 'Was the Search-refresh issue fixed?',
     'expected_tool': 'support_search',
     'expected': 'No validated resolution exists; abstain from inventing a fix.'},
]

# Sanity-check the citation case against the published data.
cited = session.sql("SELECT DISTINCT e.value::VARCHAR AS ID FROM RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES i, LATERAL FLATTEN(i.EVIDENCE_IDS) e WHERE i.THREAD_ID = 'T-100'").collect()
unresolved_fix = session.sql("SELECT COUNT(*) AS N FROM RACO_RLM_DEMO.PUBLIC.VALIDATED_ISSUES WHERE THREAD_ID = 'T-200'").collect()[0]['N']
print('T-100 published evidence IDs:', sorted(r['ID'] for r in cited) or 'none (run the publish loop first)')
print('T-200 validated issues (expect 0):', unresolved_fix)
for case in CANDIDATE_SENSE_EVALS:
    print(f"- [{case['expected_tool']}] {case['question']}")

**Checkpoint, not run by this notebook:** use the `/cortex-sense` setup workflow to discover existing contexts (don't create a duplicate), present the scope above, then the routing description, and build only after each is approved. Afterwards, merge the Sense tool into the deployed agent using `support_agent_sense.example.yaml` as a template: read the live spec and versions, back them up, show the full diff, and only then `ALTER AGENT ... MODIFY LIVE VERSION`. Keep `support_agent.agent.yaml` as the Search-only baseline. Sense can widen SQL access, so keep caller roles least-privileged. If the Sense tool isn't enabled or its documents are empty, report that rather than claiming integration.

## Support ontology proposal (see the guide's "Propose a Support Ontology" section)

**Unapproved draft, steward review required.** Nothing here imports, approves, associates or binds a Business Ontology domain.

| Proposed node | Kind | Meaning | Candidate binding |
| --- | --- | --- | --- |
| Support ticket | Entity | One seeded support case (public CSV rows are single-turn records, not threads) | `SUPPORT_MESSAGES.THREAD_ID` |
| Resolution status | Dimension | resolved / unresolved / escalated in the *validated* result, not the CSV `Issue_Status` | `THREAD_RESULTS.STATUS` |
| Evidence message | Entity | An existing message in the same seeded thread | `SUPPORT_MESSAGES.MESSAGE_ID` |
| Confirmed fix | Entity | Resolution from a thread classified resolved, with matching evidence IDs | `VALIDATED_ISSUES`, `ISSUE_CHUNKS` |

Relationships for review: `Resolution status CLASSIFIES Support ticket`, `Support ticket HAS_PART Evidence message`, and optionally `Support ticket IDENTIFIED_BY Thread ID`. "A confirmed fix needs source evidence" is an **application validation rule** (`support.py`, `validated_issue`), not a generic edge. The steward may model it as a policy node that `APPLIES_TO Confirmed fix`. There are no causal or `DERIVES` edges, and no public CSV bindings.

In [ ]:
PROPOSED_NODES = {
    'Support ticket': ('SUPPORT_MESSAGES', 'THREAD_ID'),
    'Resolution status': ('THREAD_RESULTS', 'STATUS'),
    'Evidence message': ('SUPPORT_MESSAGES', 'MESSAGE_ID'),
    'Confirmed fix': ('VALIDATED_ISSUES', 'THREAD_ID'),
}
PROPOSED_EDGES = [
    ('Resolution status', 'CLASSIFIES', 'Support ticket'),
    ('Support ticket', 'HAS_PART', 'Evidence message'),
]

columns = {(r['TABLE_NAME'], r['COLUMN_NAME']) for r in session.sql(
    "SELECT TABLE_NAME, COLUMN_NAME FROM RACO_RLM_DEMO.INFORMATION_SCHEMA.COLUMNS WHERE TABLE_SCHEMA = 'PUBLIC'").collect()}
for node, (table, column) in PROPOSED_NODES.items():
    if (table, column) not in columns:
        print(f'{node}: binding {table}.{column} NOT FOUND, do not bind')
        continue
    n = session.sql(f'SELECT COUNT(DISTINCT {column}) AS N FROM {NAMESPACE}.{table}').collect()[0]['N']
    print(f'{node}: {table}.{column} exists, {n} distinct values')

# Edge sanity checks on live data (candidate semantics, not governance).
statuses = sorted(r['STATUS'] for r in session.sql(f'SELECT DISTINCT STATUS FROM {NAMESPACE}.THREAD_RESULTS').collect())
unclassified = session.sql(f'SELECT COUNT(DISTINCT m.THREAD_ID) AS N FROM {NAMESPACE}.SUPPORT_MESSAGES m LEFT JOIN {NAMESPACE}.THREAD_RESULTS r ON r.THREAD_ID = m.THREAD_ID WHERE r.THREAD_ID IS NULL').collect()[0]['N']
fix_without_overlap = session.sql(f"""
    SELECT COUNT(*) AS N FROM {NAMESPACE}.VALIDATED_ISSUES i JOIN {NAMESPACE}.THREAD_RESULTS r ON r.THREAD_ID = i.THREAD_ID
    WHERE ARRAY_SIZE(ARRAY_INTERSECTION(i.EVIDENCE_IDS, r.EVIDENCE_IDS)) = 0""").collect()[0]['N']
print('CLASSIFIES: statuses', statuses, '| tickets without a status:', unclassified)
print('Confirmed-fix rule: issues without overlapping classification evidence (expect 0):', fix_without_overlap)
assert set(statuses) <= {'resolved', 'unresolved', 'escalated'}, 'Status outside closed set'
assert fix_without_overlap == 0, 'Confirmed fix lacks shared evidence'

**Checkpoint, not run by this notebook:** a data steward selects an existing ontology domain or approves a new one in the Business Ontology workflow, checks for existing nodes, and approves the wording before any draft, association or binding. Bind only the real assets confirmed above. Don't describe ontology terms as part of Sense retrieval until a completed build's documents show them. For relationship questions, use the governed ontology walk, not Search chunks.